[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/encuentros/e01_usar_un_modelo_existente.ipynb)

# Encuentro 1 · Usar un modelo existente y reconocer qué habría que adaptar

**Laboratorio de IA · nivel intermedio · Semana 1**

Propósito del encuentro: *reconocer tareas de lenguaje o visión donde un modelo existente pueda adaptarse y comprobarse.*

Este notebook trae dos ejemplos pequeños:

| Ejemplo | Tarea | Modelo existente | Datos |
|---|---|---|---|
| A | Clasificar mensajes de soporte por motivo | `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (zero-shot, licencia MIT) | 8 mensajes **ficticios** escritos para la clase |
| B | Clasificar fotografías de objetos | `MobileNetV2` de Keras con pesos de ImageNet | 5 fotografías de Wikimedia Commons con **licencia verificada** |

**Cómo trabajar cada ejemplo:** primero **predice** (escribe qué crees que responderá el modelo), después **ejecuta y observa**, y al final **compara** lo esperado con lo observado.

> **Ejecutado, preparado y pendiente no son lo mismo.** Si el entorno falla, el material de la clase trae la salida de una ejecución previa, rotulada con fecha y versiones. Esa salida sirve para razonar, pero no reemplaza tu propia ejecución.

## 0 · Entorno

Colab suele traer preinstaladas `tensorflow`, `transformers`, `torch` y `pillow`. Si alguna falta, quita el comentario de la línea `!pip` de la celda siguiente.

In [ ]:
# En un entorno sin estas bibliotecas, quita el comentario de la línea siguiente:
# !pip install -q tensorflow transformers torch pillow

import json, os, platform, datetime, io, urllib.request
import numpy as np
import tensorflow as tf
import transformers, torch

ENTORNO = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": tf.keras.__version__ if hasattr(tf.keras, "__version__") else "incluido en TF",
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "maquina": os.environ.get("LAB_MAQUINA", platform.platform()),
    "fecha": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
}
SALIDAS = {"encuentro": 1, "entorno": ENTORNO}
ENTORNO

## A · Clasificar mensajes con un modelo existente

**Situación ficticia.** Un equipo de soporte recibe mensajes y quiere sugerir el motivo de cada uno para que una persona lo revise. Las cuatro categorías son *facturación*, *acceso a la cuenta*, *envío* y *devolución*.

El modelo **no fue entrenado con estas categorías**. Es un modelo de inferencia en lenguaje natural (NLI): para cada categoría evalúa si la frase *«Este mensaje trata sobre {categoría}.»* se sigue del mensaje. Así se usa un modelo existente sin adaptarlo.

La columna `referencia` la escribió una persona. En un caso la propia referencia es discutible, y eso también se conversa.

In [ ]:
ETIQUETAS = ["facturación", "acceso a la cuenta", "envío", "devolución"]
PLANTILLA = "Este mensaje trata sobre {}."

MENSAJES = [
    {"id": "m1", "mensaje": "Me cobraron dos veces la misma compra este mes.", "referencia": "facturación"},
    {"id": "m2", "mensaje": "No puedo iniciar sesión: dice que mi contraseña es incorrecta.", "referencia": "acceso a la cuenta"},
    {"id": "m3", "mensaje": "El paquete lleva diez días en camino y todavía no llega.", "referencia": "envío"},
    {"id": "m4", "mensaje": "Quiero devolver los zapatos porque me quedaron pequeños.", "referencia": "devolución"},
    {"id": "m5", "mensaje": "No quiero devolver nada, solo saber cuándo llega mi pedido.", "referencia": "envío",
     "nota": "Contiene una negación: menciona «devolver» para decir que NO quiere devolver."},
    {"id": "m6", "mensaje": "Me llegó un código de verificación que yo no pedí.", "referencia": "acceso a la cuenta"},
    {"id": "m7", "mensaje": "La caja llegó abierta y falta uno de los productos.", "referencia": "envío",
     "nota": "Referencia discutible: otra persona podría etiquetarlo como devolución."},
    {"id": "m8", "mensaje": "Cambié de tarjeta y necesito actualizar el medio de pago.", "referencia": "facturación"},
]
for m in MENSAJES:
    print(f"{m['id']}  [{m['referencia']}]  {m['mensaje']}")

### Predice antes de ejecutar

Elige **dos mensajes**: uno donde esperas que el modelo acierte y otro donde crees que puede fallar. Escribe por qué.

In [ ]:
MI_PREDICCION = {
    "acierta": "m?  porque ...",
    "puede_fallar": "m?  porque ...",
}

### Ejecuta y observa

La primera ejecución descarga el modelo (~1 GB). La revisión queda fijada para que el resultado se pueda repetir.

In [ ]:
from transformers import pipeline, AutoTokenizer

MODELO_TEXTO = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
REVISION = "8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c"

clasificador = pipeline("zero-shot-classification", model=MODELO_TEXTO, revision=REVISION, device=-1)

casos_texto = []
for m in MENSAJES:
    r = clasificador(m["mensaje"], candidate_labels=ETIQUETAS, hypothesis_template=PLANTILLA)
    puntajes = {l: round(float(s), 4) for l, s in zip(r["labels"], r["scores"])}
    pred = r["labels"][0]
    casos_texto.append({**m, "prediccion": pred, "puntajes": puntajes, "acierto": pred == m["referencia"]})
    marca = "✓" if pred == m["referencia"] else "✗"
    print(f"{marca} {m['id']}  referencia={m['referencia']:<20} predicción={pred:<20} ({puntajes[pred]:.2f})")

aciertos = sum(c["acierto"] for c in casos_texto)
print(f"\nAciertos: {aciertos} de {len(casos_texto)} (muestra ficticia de 8 mensajes: es exploratorio, no una evaluación)")

### Compara: ¿dónde está el error?

Para cada error, separa tres pasos de la cadena:

1. **Preparación de la entrada**: qué texto recibe el modelo. Aquí se combina el mensaje con la plantilla *«Este mensaje trata sobre {}.»*.
2. **Representación**: cómo se convierte el texto en *tokens* y luego en números.
3. **Decisión del modelo**: qué categoría obtiene el mayor puntaje.

Después pregunta qué dato te ayudaría a saber si el error está en **la etiqueta** (la referencia es discutible), en **el contexto** (al mensaje le falta información) o en **una limitación del modelo** (resuelve otra tarea o no maneja bien algo como la negación).

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO_TEXTO, revision=REVISION)
ejemplo = next(m for m in MENSAJES if m["id"] == "m5")["mensaje"]
tokens = tok.tokenize(ejemplo)
print(ejemplo)
print(len(tokens), "tokens:", tokens)

In [ ]:
SALIDAS["texto"] = {
    "modelo": MODELO_TEXTO,
    "revision": REVISION,
    "licencia": "MIT",
    "plantilla": PLANTILLA,
    "etiquetas": ETIQUETAS,
    "casos": casos_texto,
    "aciertos": aciertos,
    "total": len(casos_texto),
    "tokens_ejemplo": {"id": "m5", "mensaje": ejemplo, "tokens": tokens},
}

## B · Clasificar fotografías con un modelo existente

**MobileNetV2** es una red convolucional (CNN) entrenada con ImageNet, que tiene **1000 clases**. Es un ejemplo del módulo M1 (*redes convolucionales aplicadas a visión artificial, en su forma básica*). Como en el ejemplo A, se usa tal cual, sin adaptar.

Las fotografías son de Wikimedia Commons; la licencia de cada una se verificó en su página y se cita abajo. Para que la descarga sea estable, el notebook las toma de una copia en el repositorio del laboratorio.

In [ ]:
IMAGENES = [
    {"id": "v1", "nombre": "taza", "referencia": "taza", "aceptables": ["coffee_mug", "cup"],
     "pagina": "https://commons.wikimedia.org/wiki/File:Coffee_Mug_(Unsplash).jpg", "autor": "Ryan Holloway hollowaykryan", "licencia": "CC0"},
    {"id": "v2", "nombre": "banana", "referencia": "banana", "aceptables": ["banana"],
     "pagina": "https://commons.wikimedia.org/wiki/File:SingleYelakki_banana,_bangalore,_karnataka_single.jpg", "autor": "Mohanramus", "licencia": "CC BY 4.0"},
    {"id": "v3", "nombre": "mochila en su empaque", "referencia": "mochila", "aceptables": ["backpack"],
     "pagina": "https://commons.wikimedia.org/wiki/File:Peak_Design_Everyday_Backpack_30_L.jpg", "autor": "Moktarama", "licencia": "CC BY 3.0",
     "nota": "La mochila aparece dentro de su caja, junto al empaque."},
    {"id": "v4", "nombre": "tetera", "referencia": "tetera", "aceptables": ["teapot"],
     "pagina": "https://commons.wikimedia.org/wiki/File:Teapot_MET_DP207945.jpg", "autor": "American Pottery Manufacturing Company (foto donada por The Met)", "licencia": "CC0"},
    {"id": "v5", "nombre": "arepa de queso", "referencia": "arepa", "aceptables": [],
     "pagina": "https://commons.wikimedia.org/wiki/File:Arepa_de_queso_by_jslander.png", "autor": "Jess Lander from Culver City, CA, USA", "licencia": "CC BY 2.0",
     "nota": "ImageNet no tiene una clase «arepa»."},
]
# Copia de las fotos en el repositorio del laboratorio (las licencias permiten redistribuirlas
# con crédito). Se descargan una vez; si ya están en la carpeta imagenes/, no se vuelven a bajar.
from PIL import Image
FOTOS = "https://raw.githubusercontent.com/Fedgutcor/lab-ia-intermedio/main/imagenes/e01/"
os.makedirs("imagenes", exist_ok=True)
for im in IMAGENES:
    ruta = os.path.join("imagenes", im["id"] + ".jpg")
    if not os.path.exists(ruta):
        datos = urllib.request.urlopen(FOTOS + im["id"] + ".jpg", timeout=60).read()
        Image.open(io.BytesIO(datos)).convert("RGB").save(ruta, quality=88)
    im["archivo"] = ruta
    print(im["id"], im["nombre"], "·", im["licencia"], "·", im["autor"])

### Predice antes de ejecutar

ImageNet tiene clases como `coffee_mug`, `banana`, `backpack` y `teapot`, pero **no** tiene `arepa`. ¿Qué crees que responderá el modelo con la arepa? ¿Y con la mochila que todavía está en su empaque?

In [ ]:
MI_PREDICCION_VISION = {
    "arepa": "...",
    "mochila_en_empaque": "...",
}

### Ejecuta y observa

Fíjate en la **preparación de la entrada**: cada foto se redimensiona a 224 × 224 píxeles y se normaliza como lo espera MobileNetV2. Si cambias esa preparación, cambia lo que ve el modelo.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

modelo = MobileNetV2(weights="imagenet")
print(f"Capas: {len(modelo.layers)} · Parámetros: {modelo.count_params():,} · Entrada: {modelo.input_shape} · Salida: {modelo.output_shape}")

casos_vision = []
for im in IMAGENES:
    x = np.array(Image.open(im["archivo"]).convert("RGB").resize((224, 224)), dtype="float32")[None, ...]
    probs = modelo.predict(preprocess_input(x), verbose=0)
    top3 = [{"clase": c, "prob": round(float(p), 4)} for (_, c, p) in decode_predictions(probs, top=3)[0]]
    acierto = top3[0]["clase"] in im["aceptables"]
    casos_vision.append({k: im[k] for k in ("id", "nombre", "referencia", "aceptables", "pagina", "autor", "licencia", "archivo")}
                        | ({"nota": im["nota"]} if "nota" in im else {}) | {"top3": top3, "acierto": acierto})
    marca = "✓" if acierto else "✗"
    print(f"{marca} {im['id']} {im['nombre']:<22} →", ", ".join(f"{t['clase']} ({t['prob']:.2f})" for t in top3))

### Compara

- ¿El error de la arepa está en la etiqueta, en el contexto o en una limitación del modelo?
- ¿Qué pasó con la mochila en su empaque? ¿El contexto de la foto ayudó o confundió al modelo?
- ¿Qué dato necesitarías para saber si el modelo serviría en *tu* proyecto?

In [ ]:
SALIDAS["vision"] = {
    "modelo": "MobileNetV2 (Keras) · pesos ImageNet",
    "capas": len(modelo.layers),
    "parametros": int(modelo.count_params()),
    "entrada": list(modelo.input_shape[1:]),
    "salida": int(modelo.output_shape[-1]),
    "casos": casos_vision,
}

## C · ¿Qué tendría que cambiar para adaptarlo?

Usar un modelo existente es una cosa y adaptarlo es otra. Si tu proyecto necesita **tus propias categorías**, por ejemplo cuatro, el esquema más común es este:

- conservar la **base** que ya aprendió a extraer patrones, con sus pesos **congelados**;
- reemplazar la **salida** de 1000 clases por una nueva con tus categorías, la única parte que se entrena.

La celda siguiente **solo construye** ese esquema y cuenta sus parámetros. **No entrena nada**: para entrenarlo hacen falta datos etiquetados, una partición y una evaluación, que son los temas de los próximos encuentros.

In [ ]:
CATEGORIAS_PROPIAS = ["categoría 1", "categoría 2", "categoría 3", "categoría 4"]

base = MobileNetV2(weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))
base.trainable = False
adaptado = tf.keras.Sequential([base, tf.keras.layers.Dense(len(CATEGORIAS_PROPIAS), activation="softmax")])
adaptado.build((None, 224, 224, 3))

entrenables = int(sum(np.prod(v.shape) for v in adaptado.trainable_weights))
congelados = int(sum(np.prod(v.shape) for v in adaptado.non_trainable_weights))
print(f"Parámetros congelados (base): {congelados:,}")
print(f"Parámetros entrenables (salida nueva): {entrenables:,}")
print(f"Proporción entrenable: {entrenables / (entrenables + congelados):.4%}")

SALIDAS["adaptacion"] = {
    "categorias": len(CATEGORIAS_PROPIAS),
    "congelados": congelados,
    "entrenables": entrenables,
    "entrenado": False,
}

## Guardar las salidas

El archivo `e01_salidas.json` conserva lo que se ejecutó, con fecha y versiones, para poder citarlo como evidencia.

In [ ]:
with open("e01_salidas.json", "w", encoding="utf-8") as f:
    json.dump(SALIDAS, f, ensure_ascii=False, indent=2)
print("Guardado e01_salidas.json ·", ENTORNO["fecha"])

## Para tu proyecto (sin tarea adicional)

Antes del próximo encuentro conserva **dos oportunidades** con usuario, modalidad (lenguaje o visión) y error que importa, más **una condición de viabilidad** (recursos o datos). En el Encuentro 2 se precisa el *brief* y se obtiene una **primera inferencia de referencia** antes de ajustar parámetros.

Módulo relacionado: **M1 · Fundamentos de redes neuronales y Deep Learning** (semana 1).